In [1]:
# Get tide from TWL forecast

import os
import time
import utils_runup
import requests
import pandas as pd
from datetime import datetime

startT= '2024-12-09'
endT = '2025-03-14'
hourly = pd.date_range(start=startT, end=endT, freq='D')
formatted_time = hourly.strftime("%Y-%m-%d")

timestamps = formatted_time
twl_region = 1
siteid = 1548
lat, lon = 41.897136303912966, -69.96233057531904
savefolder = "../DATA/data_wl"

records = []
print('getting tide')

for timestamp in timestamps:
    try:
        d = utils_runup.fetch_water_level_data(
            timestamp=timestamp,
            region_choice=twl_region, 
            site_id=siteid,
            lat=lat, 
            lon=lon,
            distance_m=500, 
            save_folder=savefolder
        )
        
        # FIX: Check if data was actually returned before trying to look inside it
        if d is None:
            print(f"Skipping {timestamp} due to data retrieval failure.")
            continue
            
        data_timestamps = pd.to_datetime(d['dateTime'], utc=True)
        TWL_forecast = d.items()
        target_timestamp = pd.to_datetime(timestamp, utc=True)
        
        # Append data if successful
        records.append(d)
        
    except Exception as e:
        print(f"Error processing {timestamp}: {e}")
        continue
        
    # Keep the 1-second pacing to avoid hitting future 429 limits
    time.sleep(1.0)  
    closest_index = abs(data_timestamps - target_timestamp).argmin()
    
    TWL_forecast = {key: values[closest_index] for key, values in d.items()}
    tide = TWL_forecast['tideWindSetup']
    
    url = "https://vdatum.noaa.gov/vdatumweb/api/convert"

    params = {
        "s_x": lon,
        "s_y": lat,
        "region": "CONTIGUOUS",
        "s_h_frame": "NAD83_2011",
        "s_h_zone": "",
        "s_coor": "geo",
        "s_h_unit": "",
        "s_v_frame": "LMSL",
        "s_v_unit": "m",
        "s_v_elevation": "height",
        "s_v_geoid": "geoid18",
        "s_z": "0.0",
        "t_h_frame": "NAD83_2011",
        "t_coor": "utm",
        "t_h_unit": "m",
        "t_h_zone": "19",
        "t_v_frame": "NAVD88",
        "t_v_unit": "m",
        "t_v_elevation": "height",
        "t_v_geoid": "geoid18",
        "epoch_in": "0.0",
        "epoch_out": "0.0"
                }


    response = requests.get(url, params=params)
    vdatum_data = response.json()

    t_z = float(vdatum_data.get("t_z"))
    tide = tide + t_z
    TWL_forecast['twl'] += t_z
    TWL_forecast['twl05'] += t_z
    TWL_forecast['twl95'] += t_z
    TWL_forecast['tideWindSetup'] += t_z

    records.append(TWL_forecast)

df = pd.DataFrame(records)
print(df.head(15))
df.to_csv(os.path.join(savefolder, f"tide_{siteid}_{startT}-{endT}.csv"), index=False)

getting tide
Getting water level forecast for Boston, MA, site ID 1548, timestamp 2024-12-09
Water level data saved to ../DATA/data_wl\Boston,_MA_1548_2024-12-09.json
Getting water level forecast for Boston, MA, site ID 1548, timestamp 2024-12-10
Water level data saved to ../DATA/data_wl\Boston,_MA_1548_2024-12-10.json
Getting water level forecast for Boston, MA, site ID 1548, timestamp 2024-12-11
Water level data saved to ../DATA/data_wl\Boston,_MA_1548_2024-12-11.json
Getting water level forecast for Boston, MA, site ID 1548, timestamp 2024-12-12
Water level data saved to ../DATA/data_wl\Boston,_MA_1548_2024-12-12.json
Getting water level forecast for Boston, MA, site ID 1548, timestamp 2024-12-13
Water level data saved to ../DATA/data_wl\Boston,_MA_1548_2024-12-13.json
Getting water level forecast for Boston, MA, site ID 1548, timestamp 2024-12-14
Water level data saved to ../DATA/data_wl\Boston,_MA_1548_2024-12-14.json
Getting water level forecast for Boston, MA, site ID 1548, time

In [ ]:
import matplotlib.pyplot as plt

plt.plot(d['dateTime'], d['twl'], label='TWL')
plt.xlabel('Date')
plt.ylabel('TWL')
plt.title('Tide Water Level Forecast')
plt.legend()
plt.show()
